# 01 · Collection: public procurement contracts (BASE)

**Goal:** get every public contract for one municipality (Covilhã) into `data/raw/`, in a way that can be re-run daily.

**Source:** IMPIC, which runs the BASE portal, publishes the complete BASE contract dataset on
[dados.gov.pt](https://dados.gov.pt/pt/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/),
with one zipped JSON file per year (2012 onwards).

**Why not scrape base.gov.pt pages?** The earlier prototype drove Chrome with Selenium through the BASE search page.
As of 2026-09-27, that page's backend call (`POST /Base4/pt/resultados/`) returns **HTTP 404** even inside a real
browser, and plain HTTP clients are blocked by the site's F5 firewall. The open-data files hold the same records with
more fields (NIFs, all bidders, base price) and need no browser at all.

Re-running this notebook downloads a year's file again only if IMPIC has republished it.

**Runs in Google Colab or locally.** The first code cell detects the environment. In Colab it mounts Google Drive and keeps everything under `My Drive/procurement-poc/data/`, so results survive the session.

In [1]:
# ── Environment: Google Colab or local (VS Code / Jupyter) ─────────────────────────────
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
DRIVE_FOLDER = "procurement-poc"   # folder in "My Drive" that holds the project data
PIP_PACKAGES = []

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = Path("/content/drive/MyDrive") / DRIVE_FOLDER / "data"   # persistent: survives the session
    WORK = Path("/content/work")                                     # fast local disk: wiped when the session ends
    if PIP_PACKAGES:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES], check=True)
else:
    DATA = Path("data")
    WORK = DATA

DATA.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
print(f"running in Colab: {IN_COLAB} | data folder: {DATA}")


Mounted at /content/drive
running in Colab: True | data folder: /content/drive/MyDrive/procurement-poc/data


In [2]:
import zipfile

import pandas as pd
import requests

DATASET_API = "https://dados.gov.pt/api/1/datasets/contratos-publicos-portal-base-impic-contratos-de-2012-a-2026/"

YEARS = [2026]                                         # add 2025, 2024, ... to backfill (one year at a time fits Colab's RAM)
MUNICIPALITY = "Portugal, Castelo Branco, Covilhã"     # exactly as written in the `localExecucao` field
SLUG = "covilha"                                       # used in output file names

RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

## 1. List the files available in the dataset
The dados.gov.pt API returns every resource with its URL and `last_modified` timestamp. The timestamp is what tells a daily run whether anything is new.

In [3]:
meta = requests.get(DATASET_API, timeout=30).json()
zips = {r["title"]: r for r in meta["resources"] if r["format"] == "zip"}

print(meta["title"], "| update frequency:", meta["frequency"])
pd.DataFrame(
    [{"file": t, "MB": round(r["filesize"] / 1e6, 1), "last_modified": r["last_modified"][:19]} for t, r in zips.items()]
).sort_values("file", ascending=False).head(5)

Contratos Públicos - Portal Base - IMPIC - Contratos de 2012 a 2026 | update frequency: weekly


,file,MB,last_modified
0,contratos2026.zip,45.6,2026-10-18T09:06:22
1,contratos2025.zip,55.4,2026-10-18T09:06:31
2,contratos2024.zip,44.6,2026-10-18T09:06:39
3,contratos2023.zip,39.0,2026-10-18T09:06:46
4,contratos2022.zip,34.8,2026-10-18T09:06:52


## 2. Download (cached)
The file name includes IMPIC's `last_modified` stamp. If that exact file is already on disk, the download is skipped; if IMPIC has published a newer version, the older copy for that year is replaced.

In [4]:
def download_year(year: int) -> Path:
    res = zips[f"contratos{year}.zip"]
    stamp = res["last_modified"][:19].replace("-", "").replace(":", "")
    path = RAW / f"contratos{year}_{stamp}.zip"
    if path.exists():
        print(f"{path.name}: up to date, skipping download")
        return path

    for old in RAW.glob(f"contratos{year}_*.zip"):
        old.unlink()
    with requests.get(res["url"], stream=True, timeout=300) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)
    print(f"{path.name}: downloaded {path.stat().st_size / 1e6:.1f} MB")
    return path


zip_paths = {year: download_year(year) for year in YEARS}

contratos2026_20261018T090622.zip: downloaded 45.6 MB


## 3. Load and filter to the municipality
Each zip holds one JSON array covering the whole country for that year (about 180k contracts for 2026). The filter keeps a
contract if the municipality appears in any of its execution locations (`localExecucao`), the same criterion the BASE search form used (país / distrito / concelho).

In [5]:
def load_year(path: Path) -> pd.DataFrame:
    with zipfile.ZipFile(path) as z, z.open(z.namelist()[0]) as f:
        return pd.read_json(f, dtype={"idcontrato": "int64", "idprocedimento": "int64"})


frames = []
for year, path in zip_paths.items():
    df = load_year(path)
    local = df[df["localExecucao"].map(lambda locs: MUNICIPALITY in (locs or []))]
    print(f"{year}: {len(df):>7,} contracts nationally -> {len(local):,} in {MUNICIPALITY}")
    frames.append(local)

contracts = pd.concat(frames, ignore_index=True)
print(f"dropping {contracts['idcontrato'].duplicated().sum()} duplicated contract ids published by IMPIC")
contracts = contracts.drop_duplicates("idcontrato", keep="last")
del df, frames  # the national frames are large; free the memory
contracts.shape

2026: 194,863 contracts nationally -> 1,328 in Portugal, Castelo Branco, Covilhã
dropping 1 duplicated contract ids published by IMPIC


(1327, 39)

## 4. Quick look

In [6]:
pub = pd.to_datetime(contracts["dataPublicacao"], format="%d/%m/%Y", errors="coerce")
print(f"published between {pub.min():%Y-%m-%d} and {pub.max():%Y-%m-%d}")
print(f"total contract value: {contracts['precoContratual'].sum():,.2f} €")
contracts[["idcontrato", "dataPublicacao", "tipoprocedimento", "objectoContrato", "adjudicante", "adjudicatarios", "precoContratual"]].head()

published between 2026-01-07 and 2026-10-16
total contract value: 79,766,666.01 €


,idcontrato,dataPublicacao,tipoprocedimento,objectoContrato,adjudicante,adjudicatarios,precoContratual
0,12446543,07/01/2026,Concurso público,EMPREITADA DE OBRAS DE REABILITAÇÃO DE PAVIMEN...,[505330768 - Município da Covilhã],[501371974 - Amadeu Gonçalves Cura &amp;amp; F...,134286.34
1,12460933,08/01/2026,Concurso público,Aquisição de contentores de polietileno de al...,"[507611977 - ADC - Águas da Covilhã, EM]","[515980196 - Waste To Me, Lda]",58600.00
2,12467834,08/01/2026,Ajuste Direto Regime Geral,"Aquisição de serviços de espetáculo ""Espirito...",[505330768 - Município da Covilhã],[515099783 - ASSOC ORQUESTRA SEM FRONTEIRAS],8600.00
3,12543104,14/01/2026,Consulta Prévia Simplificada,AQUISIÇÃO DE SERVIÇOS DE MARKETING DIGITAL NO ...,[505330768 - Município da Covilhã],"[507081951 - MILIGRAM- DESIGN,LDA]",166620.79
4,12552446,14/01/2026,Ajuste Direto Regime Geral,Empreitada de obras de requalificação das inst...,[505330768 - Município da Covilhã],[513541152 - Trienergy - Eng. e Clim. e Energi...,19361.84


In [7]:
contracts["tipoprocedimento"].value_counts().to_frame("contracts")

,contracts
tipoprocedimento,
Ajuste Direto Regime Geral,431
Ao abrigo de acordo-quadro (art.º 259.º),425
Concurso público,245
Consulta Prévia,146
Contratação excluída II,50
Ao abrigo de acordo-quadro (art.º 258.º),15
Consulta Prévia Simplificada,9
Concurso limitado por prévia qualificação,4
Setores especiais – isenção parte II,2


## 5. Save the raw extract
Records are kept exactly as IMPIC publishes them, with list fields left as lists and nothing renamed. All cleaning happens in `02_storage.ipynb`, so the raw file can always be reprocessed.

In [8]:
out = RAW / f"contracts_{SLUG}.json"
contracts.to_json(out, orient="records", force_ascii=False, indent=1)
print(f"saved {len(contracts):,} contracts -> {out}  ({out.stat().st_size / 1e6:.1f} MB)")

saved 1,327 contracts -> /content/drive/MyDrive/procurement-poc/data/raw/contracts_covilha.json  (2.8 MB)


### Note for later (T3: document extraction)
`linkPecasProc` links to the procedure documents on the e-procurement platforms. This is where PDF/OCR extraction will pick up.

In [9]:
has_docs = contracts["linkPecasProc"].fillna("").str.startswith("http")
print(f"{has_docs.sum():,} of {len(contracts):,} contracts have a procedure-documents link")
contracts.loc[has_docs, "linkPecasProc"].head(3).tolist()

988 of 1,327 contracts have a procedure-documents link


['https://community.vortal.biz/Public/public-tender-documents/RFczbnV0MStpOUpuZERYVldSb0NhbE91TW1vekhRTjFhdnpQT2tVRkJJT1FZZ2M2cFVSRjBMVHRrZlpQV1RNS3Blc2g1SVRraGtwQ3d0TkJ0bGNvUVE9PVBSUXpA',
 'https://community.vortal.biz/Public/public-tender-documents/emZ0MUltUVJxK1FVQ0E2akZBTGFVZmF4cURMTkRKSDVZY09Mei9jVnFkWkM1WHdXWHI3bzdxTUxSdzBqVjBFRFJ2SEd0QmdORmZJYmllNWl6Tk84WFE9PXdyQyR2',
 'https://community.vortal.biz/Public/public-tender-documents/OWREZWpidlBla3Yxb1I4ZnlGVFhOWGthdFpCL3prTkZYN2RpTGhiYk1uSzFXVVpLUlBHL2tLOHRueTgxVzgwOXpIQTdjVG9LMStXV0Y0Z29vNnJaTGc9PTE3R3Nz']